# Advanced Chunking Strategies

This file teaches you chunking methods that use AI to decide where to split.
By the end, you will know how to do semantic chunking, parent-child chunking, contextual chunking, proposition-based chunking, and agentic chunking.

## Setup

We need:
- **OpenAI client**: for embeddings and LLM calls.
- **numpy**: for calculating similarity between embeddings.

Key terms:
- **Embedding**: a list of numbers that represents the meaning of a piece of text.
- **Cosine similarity**: a number between -1 and 1 that measures how similar two embeddings are. 1 means identical meaning, 0 means unrelated.

In [4]:
!pip install -q openai numpy python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import re
import json
import numpy as np
from openai import OpenAI
import os

In [5]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [6]:
client = OpenAI(api_key=api_key)

In [7]:
MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

**What happened:** We set up the OpenAI client and chose the models we will use.

### Sample Document

In [94]:
SAMPLE_DOC = """Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match.

The Pod is the smallest deployable unit in Kubernetes. A Pod can contain one or more containers that share storage and network. Pods are ephemeral.

Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics. It checks metrics every 15 seconds by default.

Kubernetes networking uses a flat network model where every Pod gets its own IP address. Services provide stable endpoints for accessing Pods.

Persistent Volumes (PVs) provide durable storage that survives Pod restarts. PersistentVolumeClaims (PVCs) are requests for storage by Pods.

Security in Kubernetes involves RBAC for authorization, Network Policies for traffic control, and Pod Security Standards for container restrictions."""

In [95]:
print(len(SAMPLE_DOC))

866


**What happened:** We created a sample document about Kubernetes with 6 paragraphs, each on a different topic.

### Helper Functions

In [20]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    
    embeddings = []

    for item in response.data:
        embeddings.append(item.embedding)

    return embeddings

In [22]:
def cosine_similarity(a, b):
    
    a, b = np.array(a), np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

**What happened:** We created two helper functions. `get_embeddings` turns text into number lists. `cosine_similarity` measures how similar two number lists are.

## Strategy 1: Semantic Chunking

**What it does:** Splits the text where the meaning changes. It embeds each sentence, compares consecutive sentences, and splits wherever the similarity drops below a threshold.

**When to use it:** When the document covers multiple topics and you want each chunk to contain exactly one topic.

**Steps:**
1. Split the text into sentences.
2. Embed every sentence.
3. Compare each sentence's embedding to the next sentence's embedding.
4. Where similarity drops below the threshold, start a new chunk.

In [50]:
def semantic_chunk(text, threshold=0.75):

    raw_sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    print("RAW_SENTENCES:\n", raw_sentences, "\n")


    sentences = []
    for i, sentence in enumerate(raw_sentences):
        print(f"LENGTH OF SENTENCE {i+1}:", len(sentence))

        if len(sentence) > 10:
            sentences.append(sentence)
    print(f"\nSENTENCES LENGTH: {len(sentences)}\n")


    if len(sentences) <= 1:
        return [text]
    

    embeddings = get_embeddings(sentences)
    
    chunks = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        
        sim = cosine_similarity(embeddings[i-1], embeddings[i])

        if sim < threshold:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    
    if current:
        chunks.append(" ".join(current))

    return chunks

In [51]:
chunks = semantic_chunk(SAMPLE_DOC, threshold=0.75)

for i, chunk in enumerate(chunks):
    print(f"CHUNK {i}: {chunk}")

RAW_SENTENCES:
 ['Kubernetes uses a declarative model.', 'You describe the desired state in YAML manifests, and controllers make the actual state match.', 'The Pod is the smallest deployable unit in Kubernetes.', 'A Pod can contain one or more containers that share storage and network.', 'Pods are ephemeral.', 'Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics.', 'It checks metrics every 15 seconds by default.', 'Kubernetes networking uses a flat network model where every Pod gets its own IP address.', 'Services provide stable endpoints for accessing Pods.', 'Persistent Volumes (PVs) provide durable storage that survives Pod restarts.', 'PersistentVolumeClaims (PVCs) are requests for storage by Pods.', 'Security in Kubernetes involves RBAC for authorization, Network Policies for traffic control, and Pod Security Standards for container restrictions.'] 

LENGTH OF SENTENCE 1: 36
LENGTH OF SENTENCE 2: 94
LENGTH OF SENTENCE 3: 54
LENGTH O

**What happened:** The model embedded each sentence and compared neighbors. Where the similarity dropped (meaning the topic changed), a new chunk started.

## Strategy 2: Parent-Child Chunking

**What it does:** Creates two levels of chunks. Large **parent** chunks give the LLM enough context. Small **child** chunks are used for searching.

**When to use it:** When you want precise search results but still need to give the LLM enough surrounding text.

**Steps:**
1. Split the document into large parent chunks.
2. Split each parent into smaller child chunks.
3. Each child stores a link to its parent.
4. At search time: find the best child, then return its parent to the LLM.

In [57]:
def parent_child_chunk(text, parent_size = 500, child_size = 150):

    raw_paragraphs = text.split("\n\n")

    paragraphs = []
    for paragraph in raw_paragraphs:
        cleaned = paragraph.strip()
        if cleaned:
            paragraphs.append(cleaned)


    parents = []
    current = ""

    for paragraph in paragraphs:
        combined_length = len(paragraph) + len(current) + 2

        if combined_length <= parent_size:
            if current:
                current += paragraph
            else:
                current = paragraph
        else:
            if current:
                parents.append(current)
            current = paragraph
    
    if current: 
        parents.append(current)

    

    children = []

    for pid, parent_text in enumerate(parents):

        sentences = re.split(r'(?<=[.!?])\s+', parent_text)

        child_text = ""

        for sentence in sentences:

            if len(child_text) + len(sentence) + 1 <= child_size:
                if child_text:
                    child_text += sentence
                else:
                    child_text = sentence
            else:
                if child_text:
                    children.append({
                        "child": child_text,
                        "parent_id": pid
                    })
                child_text = sentence
            
        if child_text:
            children.append({
                "child": child_text,
                "parent_id": pid
            })
    
    return parents, children



In [58]:
parents, children = parent_child_chunk(SAMPLE_DOC)

print(len(parents))

for p in parents:
    print(p)

print(len(children))

for c in children:
    print(c)

2
Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match.The Pod is the smallest deployable unit in Kubernetes. A Pod can contain one or more containers that share storage and network. Pods are ephemeral.Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics. It checks metrics every 15 seconds by default.
Kubernetes networking uses a flat network model where every Pod gets its own IP address. Services provide stable endpoints for accessing Pods.Persistent Volumes (PVs) provide durable storage that survives Pod restarts. PersistentVolumeClaims (PVCs) are requests for storage by Pods.Security in Kubernetes involves RBAC for authorization, Network Policies for traffic control, and Pod Security Standards for container restrictions.
8
{'child': 'Kubernetes uses a declarative model.', 'parent_id': 0}
{'child': 'You describe the desired state in YAML manifests, and control

**What happened:** We split the document into large parent chunks, then split each parent into small children. Each child knows which parent it belongs to. At search time, you search the children but return the parent to the LLM.

## Strategy 3: Contextual Chunking

**What it does:** Adds a short context sentence to the beginning of each chunk. The LLM reads the full document and writes a sentence explaining what this chunk is about.

**When to use it:** When chunks contain words like "it", "they", or "this" that are unclear without the surrounding document.

**Steps:**
1. Split the document into chunks.
2. For each chunk, ask the LLM: "What is this chunk about in the context of the full document?"
3. Prepend that context to the chunk before embedding it.

In [69]:
def add_context(full_doc, chunk):

    """Ask the LLM to generate context for one chunk."""

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=100,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": 
                f"Document: \n{full_doc[:2000]}\n\n"
                f"Chunk: \n{chunk}\n\n"
                "Write 1 sentence explaining what this chunk is about in the full document."
            }
        ]
    )

    answer = response.choices[0].message.content
    answer = answer.strip()

    return answer

In [76]:
raw_paragraphs = SAMPLE_DOC.split("\n\n")

paragraphs = []

for paragraph in raw_paragraphs:
    paragraph = paragraph.strip()
    if paragraph:
        paragraphs.append(paragraph)

for i, p in enumerate(paragraphs):
    print(f"PARAGRAPH {i+1}: {p}")

PARAGRAPH 1: Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match.
PARAGRAPH 2: The Pod is the smallest deployable unit in Kubernetes. A Pod can contain one or more containers that share storage and network. Pods are ephemeral.
PARAGRAPH 3: Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics. It checks metrics every 15 seconds by default.
PARAGRAPH 4: Kubernetes networking uses a flat network model where every Pod gets its own IP address. Services provide stable endpoints for accessing Pods.
PARAGRAPH 5: Persistent Volumes (PVs) provide durable storage that survives Pod restarts. PersistentVolumeClaims (PVCs) are requests for storage by Pods.
PARAGRAPH 6: Security in Kubernetes involves RBAC for authorization, Network Policies for traffic control, and Pod Security Standards for container restrictions.


In [81]:
for paragraph in paragraphs[:2]:
    context = add_context(SAMPLE_DOC, paragraph)
    print(f"CONTEXT:{context}")
    print(f"PARAGRAPH: {paragraph}")

    combined = context + " | " + paragraph

    print("COMBINED: ", combined, "\n")

CONTEXT:This chunk explains the declarative model of Kubernetes, where users define the desired state of the system using YAML manifests, and controllers work to ensure that the actual state aligns with this definition.
PARAGRAPH: Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match.
COMBINED:  This chunk explains the declarative model of Kubernetes, where users define the desired state of the system using YAML manifests, and controllers work to ensure that the actual state aligns with this definition. | Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match. 

CONTEXT:This chunk explains the concept of Pods in Kubernetes, highlighting their role as the smallest deployable unit that can contain multiple containers sharing resources, while also noting their ephemeral nature.
PARAGRAPH: The Pod is the smallest deployable unit in Kuberne

**What happened:** The LLM read the full document and the chunk, then wrote a short summary for each chunk. This summary is added to the front of the chunk before it gets embedded, making search more accurate.

## Strategy 4: Proposition-Based Chunking

**What it does:** Extracts individual facts (**propositions**) from the text. Each proposition is one self-contained statement.

**When to use it:** When you need very precise retrieval and each fact matters on its own (legal documents, technical specs).

**Steps:**
1. Give a paragraph to the LLM.
2. Ask it to extract every individual fact.
3. Each fact becomes its own chunk.

In [84]:
def extract_propositions(text): 
    response = client.chat.completions.create(
        model = MODEL,
        max_tokens=500,
        temperature=0,
        response_format={
            "type":"json_object"
        },
        messages = [
            {
                "role": "system",
                "content": "Return JSON only."
            }, 
            {
                "role": "user",
                "content": 
                f"Extract atomic facts from this text. "
                f"Each fact should be one self-contained sentence. "
                f"Replace pronouns with the actual subject.\n\n"
                f"Text: {text}\n\n"
                f'Return: {{"propositions": ["fact1", "fact2", ...]}}'
            }
        ]
    )

    raw_content = response.choices[0].message.content
    parsed = json.loads(raw_content)

    return parsed["propositions"]

In [88]:
test_paragraph = SAMPLE_DOC.split("\n\n")[2]
print("TEST PARAGRAPH\n", test_paragraph, "\n")

propositions = extract_propositions(test_paragraph)

for i,p in enumerate(propositions):
    print(f"PREPOSITION {i+1}: {p}")

TEST PARAGRAPH
 Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics. It checks metrics every 15 seconds by default. 

PREPOSITION 1: Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics.
PREPOSITION 2: Horizontal Pod Autoscaler (HPA) checks metrics every 15 seconds by default.


**What happened:** The LLM read one paragraph and broke it into individual facts. Each fact is a complete sentence that makes sense on its own.

## Strategy 5: Agentic Chunking

**What it does:** The LLM reads the entire document and decides where to split it. It groups related sentences together by topic.

**When to use it:** When the document has mixed content and you want the highest quality chunks. This is the most expensive method because it sends the full document to the LLM.

In [97]:
def agentic_chunk(text, target_chunks = 4):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=800,
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system",
                "content": "Return JSON only."
            }, 
            {
                "role": "user",
                "content": 
                f"Split this document into {target_chunks} chunks. "
                f"Each chunk should cover ONE topic. \n\n"
                f"Document: {text}\n\n"
                f'Return: {{"chunks": [{{"topic": "name", "text": "chunk text"}}, ...]}}'
            }
        ]
    )

    raw_content = response.choices[0].message.content
    parsed = json.loads(raw_content)

    return parsed["chunks"]

In [99]:
chunks = agentic_chunk(SAMPLE_DOC, target_chunks=4)

for i, chunk in enumerate(chunks):
    print(f"CHUNK {i+1}: {chunk}\n")

CHUNK 1: {'topic': 'Declarative Model', 'text': 'Kubernetes uses a declarative model. You describe the desired state in YAML manifests, and controllers make the actual state match.'}

CHUNK 2: {'topic': 'Pods', 'text': 'The Pod is the smallest deployable unit in Kubernetes. A Pod can contain one or more containers that share storage and network. Pods are ephemeral.'}

CHUNK 3: {'topic': 'Horizontal Pod Autoscaler', 'text': 'Horizontal Pod Autoscaler (HPA) scales the number of Pods based on CPU utilization or custom metrics. It checks metrics every 15 seconds by default.'}

CHUNK 4: {'topic': 'Networking and Storage', 'text': 'Kubernetes networking uses a flat network model where every Pod gets its own IP address. Services provide stable endpoints for accessing Pods. Persistent Volumes (PVs) provide durable storage that survives Pod restarts. PersistentVolumeClaims (PVCs) are requests for storage by Pods.'}

CHUNK 5: {'topic': 'Security', 'text': 'Security in Kubernetes involves RBAC fo

**What happened:** The LLM read the full document and grouped sentences by topic. Each chunk has a topic label and contains only related content.

## When to Use What

| Strategy | Cost | Best For |
|---|---|---|
| Semantic | Low (embedding calls) | Documents with clear topic changes |
| Parent-child | Free (no API calls) | When you need precise search + full context |
| Contextual | Moderate (1 LLM call per chunk) | Chunks with unclear references |
| Proposition-based | High (1 LLM call per paragraph) | Legal docs, technical specs |
| Agentic | High (1 LLM call for full doc) | Mixed content, highest quality |

## Summary

- **Semantic chunking** splits where the meaning changes, using embeddings to detect topic boundaries.
- **Parent-child chunking** uses small chunks for searching and large chunks for giving context to the LLM.
- **Contextual chunking** adds a context sentence to each chunk so the LLM knows what "it" and "they" refer to.
- **Proposition-based chunking** extracts individual facts from the text.
- **Agentic chunking** lets the LLM decide how to split the document.
- Advanced strategies cost more (API calls) but produce higher quality chunks.
- Recommended chunk size: 200-500 tokens with 10-20% overlap.